### 1. Sistema de Línea de Espera con Dos Servidores en Paralelo


In [14]:
import numpy as np

def servidor(T, tasa_llegada, tasa_atencion):
    t = 0

    N_a = 0 #Numero de clientes que han llegado
    N_d = 0 #Numero de clientes que han salido
    
    Y = np.random.exponential(1 / tasa_llegada)

    t_a = Y #Tiempo hasta que llegue el proximo cliente
    t_d1 = float('inf')
    t_d2 = float('inf')

    A = {} #Registro de los clientes que han llegado
    D = {} #Registro de los clientes que han salido 

    fila = []
    S_1 = None #Estado del servidor 
    S_2 = None

    while True:
        t = min(t_a, t_d1, t_d2)
        if t_a > T and t_d1 == float('inf') and t_d2 == float('inf'):
            break
    #==============================================
    #Llegada de un cliente
    #==============================================
        if t == t_a and t_a <= T:   #Se revisa que el cliente llegue cuando el servidor esté abierto
            N_a += 1                #Se guarda la hora de llegada del cliente en un diccionario
            A[N_a] = t
            
            t_a = t + np.random.exponential(1/tasa_llegada) #Proxima llegada
            
            if t_d1 == float('inf'):
                S_1 = N_a
                t_d1 = t + np.random.exponential(1/tasa_atencion)
            elif t_d2 == float('inf'):                      #Se revisa si hay algún servidor disponible
                S_2 = N_a                                   # Si no, el cliente se va a la fila
                t_d2 = t + np.random.exponential(1/tasa_atencion)
            else:
                fila.append(N_a)
                
        elif t == t_a and t_a > T:      #Si el cliente llega en horario cerrado, ya no se permite llegar a nadie
            t_a = float('inf')
        

    #==============================================
    #Llegada de un cliente a servidor
    #==============================================

        elif t == t_d1:
            D[S_1] = t   #Se guarda la hora de salida del cliente en un diccionario
            N_d += 1
            
            if len(fila) > 0:
                S_1 = fila.pop(0)    # Entra el que lleva más tiempo formado
                t_d1 = t + np.random.exponential(1/tasa_atencion)
            else:
                S_1 = None
                t_d1 = float('inf')
                
        elif t == t_d2:
            D[S_2] = t #Se guarda la hora de salida del cliente en un diccionario
            N_d += 1
            
            if len(fila) > 0:
                S_2 = fila.pop(0)    # Entra el que lleva más tiempo formado
                t_d2 = t + np.random.exponential(1/tasa_atencion)
            else:
                S_2 = None
                t_d2 = float('inf')
                
    tiempos_espera = [D[i] - A[i] for i in range(1, N_a + 1)]
    promedio = np.mean(tiempos_espera)
    
    print(f"Total de llegadas: {N_a}")
    print(f"Total de salidas: {N_d}")
    print(f"Tiempo promedio en el sistema: {promedio:.4f}")


cierre = 100
Tasa_llegada = 2.0
Tasa_atencionm = 1.5
servidor(cierre, Tasa_llegada, Tasa_atencionm)   

Total de llegadas: 195
Total de salidas: 195
Tiempo promedio en el sistema: 2.0320


### 2. Estimación de Integral mediante Variables Antitéticas

Cálculo de la integral usando variable antitética. Se toma una U aleatoria y uniforme entre 0, 1 y se evalúa la funcion en U y en (1-u)

In [16]:
import numpy as np

def f_antitetica(u):            #Función para el cálculo antitetico
    return (np.exp(u**2) + np.exp((1 - u)**2)) / 2

def integral_VA(n):
    rng = np.random.default_rng()
    u = rng.uniform(0, 1, n)        #Se generan los N valores de U
    return np.mean(f_antitetica(u)) #Se evalúan estos valores aleatorios en la funcion 2^x^2

n = 10
tolerancia = 0.001      #Se definne los valores clave
limite_n = 1000

while True:
    lis = []
    
    for z in range(100):
        lis.append(integral_VA(n))      #Se agrega a la lista 100 estimadores para la integral
        
    desviacion = np.std(lis, ddof=1)        #Se calcula la desviacion estándar
    error_estandar = desviacion / np.sqrt(100)
    
    if error_estandar < tolerancia:     #Si el EE es menor que la tolerancia, se termina el programa
        break
    else:                           #Si sigue siendo mayor que la tolerancia, se repite el ciclo hasta un número establecido de veces
        n += 1
        if n > limite_n:
            print(f"Excedido el límite de n={limite_n}")
            break

print(f"Promedio estimado: {np.mean(lis):.6f}")
print(f"Error estándar alcanzado: {error_estandar:.6f}")
print(f"Tamaño de muestra (n) final necesario: {n}")

Promedio estimado: 1.461582
Error estándar alcanzado: 0.000896
Tamaño de muestra (n) final necesario: 236


### 3. Técnicas de Estadística Computacional: Bootstrap y Jackknife

Se determina la media de un conjunto de datos usando el remuestreo de bootstrap y jackknife.

In [17]:
import numpy as np

def bootstrap_media(datos):
    rng = np.random.default_rng()
    muestra = rng.choice(datos, size=len(datos), replace=True)  # Se genera una muestra con reemplazo
    return np.mean(muestra)

def jackknife_media(datos, indice):
    muestra = np.delete(datos, indice)      # Se omite el dato en la posicion 'indice'
    return np.mean(muestra)

datos = np.array([35, 42, 38, 40, 45, 37, 39, 41, 44, 36, 43, 40])
n = len(datos)

# ==========================================
# Método Bootstrap
# ==========================================
B = 1000       # Se define el numero de remuestreos
lis_boot = []

for z in range(B):
    lis_boot.append(bootstrap_media(datos))     # Se agregan los estimadores a la lista

ee_boot = np.std(lis_boot, ddof=1)      # Se calcula el error estandar directo

# ==========================================
# Método Jackknife
# ==========================================

lis_jack = []

for z in range(n):
    lis_jack.append(jackknife_media(datos, z))      # Se agregan las medias omitiendo un dato a la vez

media_jack = np.mean(lis_jack)
varianza_jack = ((n - 1) / n) * np.sum((lis_jack - media_jack)**2)  # Formula de varianza para Jackknife
ee_jack = np.sqrt(varianza_jack)        # Se calcula el error estandar



print(f"Media de los datos originales: {np.mean(datos):.4f}")

print("Bootstrap:")
print(f"Promedio estimado: {np.mean(lis_boot):.4f}")
print(f"Error estándar alcanzado: {ee_boot:.4f}\n")

print("Jackknife:")
print(f"Promedio estimado: {media_jack:.4f}")
print(f"Error estándar alcanzado: {ee_jack:.4f}")

Media de los datos originales: 40.0000
Bootstrap:
Promedio estimado: 39.9798
Error estándar alcanzado: 0.8966

Jackknife:
Promedio estimado: 40.0000
Error estándar alcanzado: 0.9129
